In [1]:
import pandas as pd
import os
import numpy as np

from sqlalchemy import create_engine

In [2]:
oss_scorecard_data_path = "https://github.com/ossf/scorecard/raw/refs/heads/main/cron/internal/data/projects.csv"
data_project = pd.read_csv(oss_scorecard_data_path)

In [3]:
# If you want to extract specific metrics into separate columns:
data_project['criticality_score'] = data_project['metadata'].str.extract(r'criticality_score:([\d.]+)')

# Convert to numeric if needed
data_project['criticality_score'] = pd.to_numeric(data_project['criticality_score'], errors='coerce')

In [4]:
df_filtered = data_project[
    ~((data_project['criticality_score'].isna() | (data_project['criticality_score'] == 0))) &
    data_project['metadata'].notna()
]

In [5]:
user=os.getenv("ECOSYSTEMS_DB_USER")
password = os.getenv("ECOSYSTEMS_DB_PASSWORD")
server = "localhost"

engine = create_engine(
        f"postgresql+psycopg2://{user}:{password}@{server}:5432/ecosystems_data_collection"
    )

In [6]:
engine.connect()

# get table "repo_names" from the database in engine

repo_names = pd.read_sql_table("repo_names", engine)
repo_names_treated = repo_names[repo_names["repo_group_id"] < 200]
repo_names_treated['repo'] = repo_names_treated['url'].str.replace('https://', '', regex=False)

# drop from df_filtered all rows where repo is in repo_names_treated
df_final = df_filtered[~df_filtered['repo'].isin(repo_names_treated['repo'])]

/var/folders/gn/33r38tj96t7f9g9zl22p9tdw0000gn/T/ipykernel_18854/2402505588.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  repo_names_treated['repo'] = repo_names_treated['url'].str.replace('https://', '', regex=False)


In [7]:
# add https:// to repo column
df_final['repository_url'] = 'https://' + df_final['repo']
#drop column 'repo' 
df_final['repo_group_id'] = 203

/var/folders/gn/33r38tj96t7f9g9zl22p9tdw0000gn/T/ipykernel_18854/1269066457.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['repository_url'] = 'https://' + df_final['repo']
/var/folders/gn/33r38tj96t7f9g9zl22p9tdw0000gn/T/ipykernel_18854/1269066457.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['repo_group_id'] = 203


In [8]:
df_final

,repo,metadata,criticality_score,repository_url,repo_group_id
9,github.com/00-Evan/shattered-pixel-dungeon,criticality_score:0.422350,0.42235,https://github.com/00-Evan/shattered-pixel-dun...,203
232,github.com/05bit/peewee-async,criticality_score:0.331150,0.33115,https://github.com/05bit/peewee-async,203
315,github.com/0ad/0ad,criticality_score:0.481240,0.48124,https://github.com/0ad/0ad,203
407,github.com/0install/0install,criticality_score:0.392460,0.39246,https://github.com/0install/0install,203
424,github.com/0k/shyaml,criticality_score:0.335860,0.33586,https://github.com/0k/shyaml,203
...,...,...,...,...,...
1310382,github.com/zzzprojects/EntityFramework-Plus,criticality_score:0.471450,0.47145,https://github.com/zzzprojects/EntityFramework...,203
1310383,github.com/zzzprojects/Eval-Expression.NET,criticality_score:0.311550,0.31155,https://github.com/zzzprojects/Eval-Expression...,203
1310384,github.com/zzzprojects/System.Linq.Dynamic,criticality_score:0.304160,0.30416,https://github.com/zzzprojects/System.Linq.Dyn...,203
1310385,github.com/zzzprojects/System.Linq.Dynamic.Core,criticality_score:0.513740,0.51374,https://github.com/zzzprojects/System.Linq.Dyn...,203


In [9]:
df_final.to_csv('../data/proc/oss_scorecard_projects.csv', index=False)
#store as .txt with repository_url separated with repo_group by a comma
df_final[['repository_url', 'repo_group_id']].to_csv('../data/proc/oss_scorecard_projects.txt', index=False, header=False)